# Kuhn Poker CFR

This notebook is for implementing and exploring Counterfactual Regret Minimization on Kuhn Poker.

**Game rules recap:**
- 3 cards: J < Q < K. Each player is dealt one.
- Both antes 1 chip. One betting round, fixed bet size of 1.
- OOP acts first. IP acts second.
- Actions: `k` check, `b` bet, `c` call, `f` fold

In [6]:
# Cell 1 - game engine (imported from game.py)
from game import CARDS, GameState, act

print("Loaded game engine from game.py")
print("Cards:", CARDS)

Loaded game engine from game.py
Cards: ['J', 'Q', 'K']


## Play a hand

In [7]:
# Cell 2 - regret storage (global tables)
regrets = {}        # (card, history_tuple) -> {action: float}
strategy_sum = {}   # (card, history_tuple) -> {action: float}

print("Initialized regret storage")
print("regrets keys:", len(regrets))
print("strategy_sum keys:", len(strategy_sum))

Initialized regret storage
regrets keys: 0
strategy_sum keys: 0


In [8]:
# Cell 3 - helper functions
# These controls keep debug output readable while you learn what CFR is doing.
DEBUG_VERBOSE_ITERS = 2
PRINT_EVERY = 1000
CURRENT_ITER = 0


def _ensure_info_set(info_set, legal_actions):
    # In CFR, each information set needs a regret table and a strategy-sum table.
    if info_set not in regrets:
        regrets[info_set] = {}
        for action in legal_actions:
            regrets[info_set][action] = 0.0
        print(f"[init] regrets[{info_set}] = {regrets[info_set]}")

    # We also store cumulative strategy so we can output average strategy at the end.
    if info_set not in strategy_sum:
        strategy_sum[info_set] = {}
        for action in legal_actions:
            strategy_sum[info_set][action] = 0.0
        print(f"[init] strategy_sum[{info_set}] = {strategy_sum[info_set]}")


def get_strategy(info_set, legal_actions):
    """Regret-matching policy for one info set."""
    # CFR chooses actions by regret matching at each info set.
    _ensure_info_set(info_set, legal_actions)

    # Only positive regret contributes to action probability in vanilla CFR.
    positive_regrets = {}
    positive_total = 0.0
    for action in legal_actions:
        regret_value = regrets[info_set][action]
        positive_regret = regret_value if regret_value > 0.0 else 0.0
        positive_regrets[action] = positive_regret
        positive_total += positive_regret

    # If no action has positive regret yet, use a uniform strategy.
    strategy = {}
    if positive_total > 0.0:
        for action in legal_actions:
            strategy[action] = positive_regrets[action] / positive_total
    else:
        uniform_probability = 1.0 / len(legal_actions)
        for action in legal_actions:
            strategy[action] = uniform_probability

    # Print the local policy so you can see regret matching happen over time.
    if CURRENT_ITER <= DEBUG_VERBOSE_ITERS:
        print(f"[iter {CURRENT_ITER}] get_strategy info_set={info_set}")
        print("  regrets:", regrets[info_set])
        print("  strategy:", strategy)

    return strategy


def get_average_strategy(info_set, legal_actions):
    """Average strategy over training iterations."""
    # Final CFR output is the average policy, not the last-iteration policy.
    _ensure_info_set(info_set, legal_actions)

    # Sum the accumulated strategy weights at this info set.
    total_weight = 0.0
    for action in legal_actions:
        total_weight += strategy_sum[info_set][action]

    # Normalize cumulative weights to turn them into probabilities.
    average_strategy = {}
    if total_weight > 0.0:
        for action in legal_actions:
            average_strategy[action] = strategy_sum[info_set][action] / total_weight
    else:
        uniform_probability = 1.0 / len(legal_actions)
        for action in legal_actions:
            average_strategy[action] = uniform_probability

    # Print averages so you can compare learned frequencies with CFR theory.
    print(
        f"[avg] info_set={info_set}, "
        f"strategy_sum={strategy_sum[info_set]}, avg={average_strategy}"
    )
    return average_strategy


print("Helper functions loaded")

Helper functions loaded


In [9]:
# Cell 4 - CFR recursion

def cfr(state, reach_oop, reach_ip):
    """
    Returns utility from OOP perspective for this game state.
    reach_oop and reach_ip are realization probabilities for each player's path.
    """
    # Indentation is only for debug readability of the game tree depth.
    depth = len(state.history)
    indent = "  " * depth

    # Base case: once terminal, return payoff from OOP perspective.
    if state.is_over:
        terminal_utility = state.winnings
        if CURRENT_ITER <= DEBUG_VERBOSE_ITERS:
            print(
                f"{indent}[terminal] history={tuple(state.history)} "
                f"oop={state.oop_card} ip={state.ip_card} utility={terminal_utility}"
            )
        return terminal_utility

    # Build the information set: current player card + public action history.
    active_player = state.active_player
    if active_player == "OOP":
        private_card = state.oop_card
    else:
        private_card = state.ip_card

    history_tuple = tuple(state.history)
    legal_actions = state.legal_actions
    info_set = (private_card, history_tuple)

    # Get current regret-matching strategy for this information set.
    strategy = get_strategy(info_set, legal_actions)

    if CURRENT_ITER <= DEBUG_VERBOSE_ITERS:
        print(
            f"{indent}[node] player={active_player} card={private_card} "
            f"history={history_tuple} reach_oop={reach_oop:.4f} reach_ip={reach_ip:.4f}"
        )

    # OOP branch: evaluate each action and update OOP regrets.
    if active_player == "OOP":
        # Add this iteration's strategy contribution for average-policy reporting.
        for action in legal_actions:
            strategy_sum[info_set][action] += reach_oop * strategy[action]

        # Recursively compute utility of each possible action.
        action_utilities = {}
        for action in legal_actions:
            next_state = act(state, action)
            next_reach_oop = reach_oop * strategy[action]
            action_utility = cfr(next_state, next_reach_oop, reach_ip)
            action_utilities[action] = action_utility

        # Node utility is expected value under current mixed strategy.
        node_utility = 0.0
        for action in legal_actions:
            node_utility += strategy[action] * action_utilities[action]

        # Counterfactual regret: action value minus node value, weighted by opponent reach.
        for action in legal_actions:
            regret_delta = action_utilities[action] - node_utility
            weighted_regret = reach_ip * regret_delta
            regrets[info_set][action] += weighted_regret
            if CURRENT_ITER <= DEBUG_VERBOSE_ITERS:
                print(
                    f"{indent}[regret OOP] action={action} delta={regret_delta:.4f} "
                    f"weighted={weighted_regret:.4f} total={regrets[info_set][action]:.4f}"
                )

        return node_utility

    # IP branch: same logic, but regret is tracked in IP's utility space.
    for action in legal_actions:
        strategy_sum[info_set][action] += reach_ip * strategy[action]

    # Compute utilities for each IP action.
    oop_action_utilities = {}
    ip_action_utilities = {}
    for action in legal_actions:
        next_state = act(state, action)
        next_reach_ip = reach_ip * strategy[action]

        oop_utility = cfr(next_state, reach_oop, next_reach_ip)
        ip_utility = -oop_utility

        oop_action_utilities[action] = oop_utility
        ip_action_utilities[action] = ip_utility

    # IP node utility is expected value of IP utilities under current strategy.
    ip_node_utility = 0.0
    for action in legal_actions:
        ip_node_utility += strategy[action] * ip_action_utilities[action]

    # IP counterfactual regret is weighted by opponent (OOP) reach.
    for action in legal_actions:
        regret_delta_ip = ip_action_utilities[action] - ip_node_utility
        weighted_regret_ip = reach_oop * regret_delta_ip
        regrets[info_set][action] += weighted_regret_ip
        if CURRENT_ITER <= DEBUG_VERBOSE_ITERS:
            print(
                f"{indent}[regret IP] action={action} delta={regret_delta_ip:.4f} "
                f"weighted={weighted_regret_ip:.4f} total={regrets[info_set][action]:.4f}"
            )

    # Convert back to OOP perspective so recursion returns one consistent value type.
    return -ip_node_utility


print("CFR function loaded")

CFR function loaded


## CFR Implementation

Start here.

In [10]:
# Cell 5 - training loop
# Each iteration sweeps all chance outcomes (all distinct card deals).
NUM_ITERS = 100_000

for iteration_index in range(1, NUM_ITERS + 1):
    # Keep iteration index global so debug prints inside cfr can reference it.
    CURRENT_ITER = iteration_index
    total_iteration_utility = 0.0

    # Loop over every ordered OOP/IP card assignment except impossible equal cards.
    for oop_card in CARDS:
        for ip_card in CARDS:
            if oop_card == ip_card:
                continue

            # Start from root state and run one CFR traversal for this deal.
            state = GameState(oop_card=oop_card, ip_card=ip_card)
            utility_oop = cfr(state, reach_oop=1.0, reach_ip=1.0)
            total_iteration_utility += utility_oop

            # Early verbose prints help map tree traversal to actual card matchups.
            if iteration_index <= DEBUG_VERBOSE_ITERS:
                print(
                    f"[iter {iteration_index}] completed hand oop={oop_card}, "
                    f"ip={ip_card}, util(OOP)={utility_oop:.4f}"
                )

    # Periodic summary lets you see training progress without printing every node forever.
    if iteration_index <= DEBUG_VERBOSE_ITERS or iteration_index % PRINT_EVERY == 0:
        print(
            f"[iter {iteration_index}] aggregate util(OOP) over all "
            f"card permutations = {total_iteration_utility:.4f}"
        )

print("Training complete")

[init] regrets[('J', ())] = {'k': 0.0, 'b': 0.0}
[init] strategy_sum[('J', ())] = {'k': 0.0, 'b': 0.0}
[iter 1] get_strategy info_set=('J', ())
  regrets: {'k': 0.0, 'b': 0.0}
  strategy: {'k': 0.5, 'b': 0.5}
[node] player=OOP card=J history=() reach_oop=1.0000 reach_ip=1.0000
[init] regrets[('Q', ('k',))] = {'k': 0.0, 'b': 0.0}
[init] strategy_sum[('Q', ('k',))] = {'k': 0.0, 'b': 0.0}
[iter 1] get_strategy info_set=('Q', ('k',))
  regrets: {'k': 0.0, 'b': 0.0}
  strategy: {'k': 0.5, 'b': 0.5}
  [node] player=IP card=Q history=('k',) reach_oop=0.5000 reach_ip=1.0000
    [terminal] history=('k', 'k') oop=J ip=Q utility=-1
[init] regrets[('J', ('k', 'b'))] = {'f': 0.0, 'c': 0.0}
[init] strategy_sum[('J', ('k', 'b'))] = {'f': 0.0, 'c': 0.0}
[iter 1] get_strategy info_set=('J', ('k', 'b'))
  regrets: {'f': 0.0, 'c': 0.0}
  strategy: {'f': 0.5, 'c': 0.5}
    [node] player=OOP card=J history=('k', 'b') reach_oop=0.5000 reach_ip=0.5000
      [terminal] history=('k', 'b', 'f') oop=J ip=Q utili

In [11]:
# Cell 6 - inspect results
print("Average strategy for (K, ()) with legal ['k', 'b']")
print(get_average_strategy(("K", ()), ["k", "b"]))

print("\nExtra sanity checks:")
print("(J, ()):", get_average_strategy(("J", ()), ["k", "b"]))
print("(Q, ()):", get_average_strategy(("Q", ()), ["k", "b"]))
print("(K, ('k',)):", get_average_strategy(("K", ("k",)), ["k", "b"]))

Average strategy for (K, ()) with legal ['k', 'b']
[avg] info_set=('K', ()), strategy_sum={'k': 67394.8938931156, 'b': 132605.10610688583}, avg={'k': 0.3369744694655756, 'b': 0.6630255305344244}
{'k': 0.3369744694655756, 'b': 0.6630255305344244}

Extra sanity checks:
[avg] info_set=('J', ()), strategy_sum={'k': 155997.12828158183, 'b': 44002.871718419796}, avg={'k': 0.7799856414079028, 'b': 0.2200143585920972}
(J, ()): {'k': 0.7799856414079028, 'b': 0.2200143585920972}
[avg] info_set=('Q', ()), strategy_sum={'k': 199997.625, 'b': 2.375}, avg={'k': 0.999988125, 'b': 1.1875e-05}
(Q, ()): {'k': 0.999988125, 'b': 1.1875e-05}
[avg] info_set=('K', ('k',)), strategy_sum={'k': 2.0, 'b': 199998.0}, avg={'k': 1e-05, 'b': 0.99999}
(K, ('k',)): {'k': 1e-05, 'b': 0.99999}


In [ ]:
# Cell 7 - measure exploitability with correct best-response logic

KUHN_NASH_VALUE_OOP = -1.0 / 18.0


def evaluate_policy_value(state, policy_dict):
    """Return the expected OOP utility when both players follow policy_dict."""
    if state.is_over:
        return state.winnings

    active_player = state.active_player
    info_set_card = state.oop_card if active_player == "OOP" else state.ip_card
    history_tuple = tuple(state.history)
    info_set = (info_set_card, history_tuple)

    policy = policy_dict.get(info_set, {})
    legal_actions = state.legal_actions
    default_probability = 1.0 / len(legal_actions)

    node_value = 0.0
    for action in legal_actions:
        action_probability = policy.get(action, default_probability)
        next_state = act(state, action)
        action_value = evaluate_policy_value(next_state, policy_dict)
        node_value += action_probability * action_value

    return node_value


def best_response_value(state, policy_dict, best_response_player):
    """
    Return expected OOP utility when one player plays best response.
    best_response_player is either "OOP" or "IP".
    """
    if state.is_over:
        return state.winnings

    active_player = state.active_player
    info_set_card = state.oop_card if active_player == "OOP" else state.ip_card
    history_tuple = tuple(state.history)
    info_set = (info_set_card, history_tuple)

    # The best-response player chooses the action that is best for them.
    if active_player == best_response_player:
        action_values = []
        for action in state.legal_actions:
            next_state = act(state, action)
            value = best_response_value(next_state, policy_dict, best_response_player)
            action_values.append(value)

        if best_response_player == "OOP":
            return max(action_values)

        # IP wants to minimize OOP utility because the game is zero-sum.
        return min(action_values)

    # The non-best-response player stays on the frozen learned policy.
    policy = policy_dict.get(info_set, {})
    legal_actions = state.legal_actions
    default_probability = 1.0 / len(legal_actions)

    node_value = 0.0
    for action in legal_actions:
        action_probability = policy.get(action, default_probability)
        next_state = act(state, action)
        action_value = best_response_value(next_state, policy_dict, best_response_player)
        node_value += action_probability * action_value

    return node_value


def compute_exploitability(policy_dict):
    """
    Measure exploitability relative to Kuhn Poker's true game value.

    OOP equilibrium value is -1/18.
    A Nash strategy has:
      - OOP best-response value equal to -1/18
      - IP best-response value (from OOP perspective) equal to -1/18
      - exploitability equal to 0
    """
    learned_profile_value_total = 0.0
    oop_best_response_total = 0.0
    ip_best_response_total = 0.0

    for oop_card in CARDS:
        for ip_card in CARDS:
            if oop_card == ip_card:
                continue

            state = GameState(oop_card=oop_card, ip_card=ip_card)

            learned_profile_value_total += evaluate_policy_value(state, policy_dict)
            oop_best_response_total += best_response_value(state, policy_dict, "OOP")
            ip_best_response_total += best_response_value(state, policy_dict, "IP")

    num_hands = len(CARDS) * (len(CARDS) - 1)
    learned_profile_value = learned_profile_value_total / num_hands
    oop_best_response_value = oop_best_response_total / num_hands
    ip_best_response_value = ip_best_response_total / num_hands

    # How much extra OOP can gain above the equilibrium game value.
    oop_exploitability = oop_best_response_value - KUHN_NASH_VALUE_OOP

    # How much extra IP can gain, converted into OOP-perspective utility.
    ip_exploitability = KUHN_NASH_VALUE_OOP - ip_best_response_value

    total_exploitability = oop_exploitability + ip_exploitability
    average_exploitability = total_exploitability / 2.0

    return {
        "nash_value_oop": KUHN_NASH_VALUE_OOP,
        "learned_profile_value": learned_profile_value,
        "oop_best_response_value": oop_best_response_value,
        "ip_best_response_value": ip_best_response_value,
        "oop_exploitability": oop_exploitability,
        "ip_exploitability": ip_exploitability,
        "total_exploitability": total_exploitability,
        "average_exploitability": average_exploitability,
    }


print("Correct exploitability functions loaded")

Exploitability functions loaded


In [ ]:
# Cell 8 - build final strategy and show convergence

# Extract final average strategy for all info sets.
final_strategy = {}
for info_set in strategy_sum:
    legal_actions = list(strategy_sum[info_set].keys())
    final_strategy[info_set] = get_average_strategy(info_set, legal_actions)

print("\n" + "=" * 60)
print("CONVERGENCE ANALYSIS")
print("=" * 60)

# Compute exploitability of the final learned strategy.
exploitability_result = compute_exploitability(final_strategy)

print(f"\nLearned profile value (OOP perspective): {exploitability_result['learned_profile_value']:.6f}")
print(f"True Kuhn Nash value for OOP:           {exploitability_result['nash_value_oop']:.6f}")
print(f"Difference from Nash value:             {exploitability_result['learned_profile_value'] - exploitability_result['nash_value_oop']:.6f}")

print(f"\nOOP best-response value:               {exploitability_result['oop_best_response_value']:.6f}")
print(f"IP best-response value:                {exploitability_result['ip_best_response_value']:.6f}")
print(f"OOP exploitability gap:                {exploitability_result['oop_exploitability']:.6f}")
print(f"IP exploitability gap:                 {exploitability_result['ip_exploitability']:.6f}")
print(f"Total exploitability:                  {exploitability_result['total_exploitability']:.6f}")
print(f"Average exploitability:                {exploitability_result['average_exploitability']:.6f}")

print("\nInterpretation:")
print("  - In Nash equilibrium, both exploitability gaps are 0.")
print("  - Lower total exploitability means your average strategy is harder to exploit.")
print("  - The learned profile value should also move toward -1/18 for OOP.")

if exploitability_result["average_exploitability"] < 0.01:
    print("\nExcellent: strategy is extremely close to Nash.")
elif exploitability_result["average_exploitability"] < 0.05:
    print("\nGood: strategy is close to Nash.")
elif exploitability_result["average_exploitability"] < 0.10:
    print("\nReasonable: strategy is converging, but not especially tight yet.")
else:
    print("\nNot close enough yet: run more iterations or inspect the implementation.")

[avg] info_set=('J', ()), strategy_sum={'k': 155997.12828158183, 'b': 44002.871718419796}, avg={'k': 0.7799856414079028, 'b': 0.2200143585920972}
[avg] info_set=('Q', ('k',)), strategy_sum={'k': 199996.1666666667, 'b': 3.833333333333333}, avg={'k': 0.9999808333333333, 'b': 1.9166666666666664e-05}
[avg] info_set=('J', ('k', 'b')), strategy_sum={'f': 155996.87828158183, 'c': 0.25}, avg={'f': 0.9999983974063962, 'c': 1.6025936038305703e-06}
[avg] info_set=('Q', ('b',)), strategy_sum={'f': 132829.5751261854, 'c': 67170.42487381391}, avg={'f': 0.6641478756309294, 'c': 0.33585212436907075}
[avg] info_set=('K', ('k',)), strategy_sum={'k': 2.0, 'b': 199998.0}, avg={'k': 1e-05, 'b': 0.99999}
[avg] info_set=('K', ('b',)), strategy_sum={'f': 0.5, 'c': 199999.5}, avg={'f': 2.5e-06, 'c': 0.9999975}
[avg] info_set=('Q', ()), strategy_sum={'k': 199997.625, 'b': 2.375}, avg={'k': 0.999988125, 'b': 1.1875e-05}
[avg] info_set=('J', ('k',)), strategy_sum={'k': 133649.41260910724, 'b': 66350.5873908931}, 

In [ ]:
# Cell 9 - detailed breakdown of corrected convergence calculations

print("\n" + "=" * 70)
print("CORRECTED EXPLOITABILITY BREAKDOWN")
print("=" * 70)

print("""
What we are measuring now:
  1. The value of your learned strategy when both players use it.
  2. How much OOP can improve by deviating to best response.
  3. How much IP can improve by deviating to best response.
  4. Total exploitability, which should go to 0 at Nash.
""")

print(f"\n1. LEARNED PROFILE VALUE: {exploitability_result['learned_profile_value']:.6f}")
print(f"   True Kuhn Nash value for OOP: {exploitability_result['nash_value_oop']:.6f}")
print("""
   This is the expected OOP payoff when both players use your learned average strategy.
   In Kuhn Poker, the true equilibrium value is -1/18 for OOP, not 0.
   So a converged strategy should have learned profile value close to -0.055556.
""")

print(f"\n2. OOP BEST-RESPONSE VALUE: {exploitability_result['oop_best_response_value']:.6f}")
print(f"   OOP exploitability gap:      {exploitability_result['oop_exploitability']:.6f}")
print("""
   This asks: if IP stays on your learned strategy, how much can OOP improve
   by switching to a perfect counter-strategy?
   In equilibrium, OOP should not be able to improve above the game value.
""")

print(f"\n3. IP BEST-RESPONSE VALUE:  {exploitability_result['ip_best_response_value']:.6f}")
print(f"   IP exploitability gap:   {exploitability_result['ip_exploitability']:.6f}")
print("""
   This asks: if OOP stays on your learned strategy, how much can IP improve
   by switching to a perfect counter-strategy?
   Because values are tracked from OOP's perspective, IP best response pushes
   the number downward by minimizing OOP utility.
""")

print(f"\n4. TOTAL EXPLOITABILITY:   {exploitability_result['total_exploitability']:.6f}")
print(f"   AVERAGE EXPLOITABILITY: {exploitability_result['average_exploitability']:.6f}")
print("""
   Total exploitability = OOP gap + IP gap.
   Average exploitability = total exploitability / 2.
   These are always non-negative in a correct implementation.
   At Nash equilibrium they are exactly 0.
""")

print("\n" + "=" * 70)
print("HOW TO READ THIS")
print("=" * 70)
print("""
If your average exploitability is small, your policy is hard to exploit.
If your learned profile value is close to -1/18, your policy also matches
Kuhn Poker's true equilibrium game value.

So the two checks you want are:
  - exploitability -> 0
  - learned profile value -> -1/18 for OOP

Those together are the cleanest evidence that CFR is converging correctly.
""")

print("\n" + "=" * 70)
print("STRATEGY STABILITY CHECK")
print("=" * 70)
print("\nYour learned strategies at key positions:")
print("(These should stabilize as iterations increase.)\n")

test_positions = [
    ("J", ()),
    ("Q", ()),
    ("K", ()),
    ("Q", ("k",)),
    ("K", ("k",)),
]

for card, history in test_positions:
    info_set = (card, history)
    if info_set in final_strategy:
        strategy = final_strategy[info_set]
        print(f"{card} after {history if history else 'start'}: {strategy}")


CALCULATING EXPLOITABILITY: DETAILED BREAKDOWN

What we're measuring:
  1. Can an opponent exploit your learned strategy by playing best response?
  2. If yes, how many chips per hand can they win?
  3. Lower exploitation = closer to Nash Equilibrium.

THE THREE METRICS:


1. OOP BEST-RESPONSE VALUE: 0.331967

   Scenario: OOP knows your strategy and plays best response against it.
             IP continues playing your learned strategy.
   Interpretation: OOP can win 0.3320 chips per hand if they exploit you.
   Calculation: For each OOP/IP card combo, OOP chooses actions to maximize
                their value against your fixed strategy, then average across
                all 6 distinct deals (J-Q, J-K, Q-J, Q-K, K-J, K-Q).


2. IP BEST-RESPONSE VALUE: 0.480014

   Scenario: IP knows your strategy and plays best response against it.
             OOP continues playing your learned strategy.
   Interpretation: IP can win 0.4800 chips per hand if they exploit you.
   Calculation: For

# Nash Equilibrium Convergence

In Kuhn Poker, CFR provably converges to a Nash Equilibrium. We measure this via **exploitability**: the value an opponent can gain by playing best response against our learned strategy while we don't deviate. As we train, exploitability should decrease toward 0.